# TickDB Market Data Streaming con Spark Structured Streaming

Consume datos en tiempo real deTickDB desde Kafka y procesa eventos de mercado.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
import time

In [ ]:
spark = SparkSession.builder \
    .appName("TickDBStreaming") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark: {spark.version}")

## Paso 1: Definir esquema del evento

In [ ]:
event_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("last_price", DoubleType(), True),
    StructField("volume_24h", DoubleType(), True),
    StructField("high_24h", DoubleType(), True),
    StructField("low_24h", DoubleType(), True),
    StructField("price_change_24h", DoubleType(), True),
    StructField("price_change_percent_24h", DoubleType(), True),
    StructField("timestamp", LongType(), True),
    StructField("event_timestamp", LongType(), True),
    StructField("market", StringType(), True)
])

## Paso 2: Leer stream desde Kafka

In [ ]:
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "tickdb-market-data") \
    .option("startingOffsets", "earliest") \
    .load()

print("Columnas disponibles:", df_kafka.columns)

## Paso 3: Parsear JSON y aplicar esquema

In [ ]:
df_parsed = df_kafka.select(
    from_json(col("value").cast("string"), event_schema).alias("data"),
    col("timestamp").alias("kafka_timestamp"),
    col("topic"),
    col("partition"),
    col("offset")
).select("data.*", "kafka_timestamp", "topic", "partition", "offset")

## Paso 4: Calcular latencia y agregar métricas

In [ ]:
df_metrics = df_parsed.withColumn(
    "latency_ms", 
    col("event_timestamp") - col("timestamp")
).withColumn(
    "processing_time",
    current_timestamp()
)

## Paso 5: Filtrar datos válidos

In [ ]:
df_valid = df_metrics.filter(
    col("symbol").isNotNull() & 
    col("last_price").isNotNull()
)

## Paso 6: Salida a consola con métricas

In [ ]:
query_console = df_valid \
    .writeStream \
    .format("console") \
    .option("truncate", "false") \
    .trigger(processingTime="5 seconds") \
    .start()

## Paso 7: Salida a Parquet con ventanas y watermarking

In [ ]:
import os
checkpoint_path = "/opt/artifacts/checkpoint"
output_path = "/opt/artifacts/output/tickdb_parquet"

os.makedirs(checkpoint_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)

In [ ]:
df_windowed = df_valid \
    .withWatermark("timestamp", "10 minutes") \
    .groupBy(
        window("timestamp", "1 minute"),
        col("market")
    ) \
    .agg(
        count("*").alias("event_count"),
        avg("last_price").alias("avg_price"),
        avg("latency_ms").alias("avg_latency_ms"),
        max("price_change_percent_24h").alias("max_change_percent")
)

In [ ]:
query_parquet = df_windowed \
    .writeStream \
    .format("parquet") \
    .option("path", output_path) \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(processingTime="10 seconds") \
    .outputMode("append") \
    .start()

print(f"✅ Guardando en: {output_path}")

## Paso 8: Métricas de rendimiento

In [ ]:
time.sleep(30)

if query_console.lastProgress:
    progress = query_console.lastProgress
    print("=== MÉTRICAS DEL ÚLTIMO MICRO-BATCH ===")
    print(f"Batch ID: {progress.get('id', 'N/A')}")
    print(f"Eventos de entrada: {progress.get('numInputRows', 0)}")
    print(f"Eventos/segundo entrada: {progress.get('inputRowsPerSecond', 0):.2f}")
    print(f"Eventos/segundo proceso: {progress.get('processedRowsPerSecond', 0):.2f}")
    print(f"Duración batch (ms): {progress.get('durationMs', {})}")
    print(f"Estado: {progress.get('sinkDescription', {}).get('state', 'N/A')}")

## Paso 9: Mostrar datos procesados

In [ ]:
df_valid.select(
    "symbol",
    "last_price",
    "volume_24h",
    "price_change_percent_24h",
    "market",
    "latency_ms"
).writeStream \
    .format("console") \
    .option("truncate", "true") \
    .trigger(processingTime="10 seconds") \
    .start()

## Paso 10: Detener streams

In [ ]:
# Descomenta para detener
# query_console.stop()
# query_parquet.stop()